# Pothole Detection — YOLOv8l Training (Kaggle)

## Single-Stage YOLOv8l Pipeline with Auto-Push Sync

| Feature | Detail |
|----------|--------|
| **Environment** | Kaggle (9h session limit) |
| **Model** | YOLOv8l · 768px · 400 epochs |
| **Time Guard** | Auto-save before 8.5h limit |
| **Resume** | `last.pt` → `best.pt` → fresh (no health-check, no deletion) |
| **NaN Safety** | AMP disabled · lr0=3e-4 · gradient clip · loss guard |
| **Auto-Push** | Checkpoints pushed to `YOLOv8l-Epochs` dataset every 5 epochs |

---

## Setup Instructions

1. Attach your pothole dataset (`pothole-detection-dataset-768`) via Add Data
2. Attach your epochs dataset (`YOLOv8l-Epochs`) via Add Data
3. Set `KAGGLE_DATASET_OWNER` in Section 2d to your Kaggle username
4. Enable GPU accelerator in Settings
5. Run All

## 1 · Environment Setup

In [ ]:
"""
Section 1 · Environment Setup
"""

from __future__ import annotations
import gc, json, os, shutil, subprocess, time, warnings, zipfile
from datetime import datetime, timezone
from pathlib import Path

ENV = "KAGGLE"
SESSION_TIME_LIMIT_H = 8.5

print("=" * 60)
print(f"  [ENV DETECTED] {ENV}")
print(f"  [SESSION LIMIT] {SESSION_TIME_LIMIT_H} hours")
print("=" * 60)

!pip install -q ultralytics

import cv2, matplotlib.pyplot as plt, numpy as np, pandas as pd
import torch, yaml
from IPython.display import display, FileLink
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# quick check to confirm the kaggle CLI is available and authenticated
import subprocess
result = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## 2 · Project Paths, Dataset & Checkpoint Import

In [ ]:
"""Section 2a · Project Paths"""

from pathlib import Path

BASE_DIR   = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")

BACKEND_DIR     = BASE_DIR / "backend"
MODELS_DIR      = BACKEND_DIR / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"
FINAL_DIR       = MODELS_DIR / "final"
LOGS_DIR        = MODELS_DIR / "logs"
V8L_DIR         = MODELS_DIR / "v8l"
INFERENCE_DIR   = BACKEND_DIR / "inference_outputs"
RUNS_DIR        = BACKEND_DIR / "runs"

for d in [MODELS_DIR, CHECKPOINTS_DIR, FINAL_DIR, LOGS_DIR,
          V8L_DIR, INFERENCE_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Scanning /kaggle/input ...")
if INPUT_ROOT.exists():
    for d in sorted(INPUT_ROOT.iterdir()):
        n = sum(1 for _ in d.rglob("*") if _.is_file())
        print(f"   {d.name}/ ({n} files)")

print("\nWorking dirs created under /kaggle/working/backend/")


In [ ]:
"""Section 2b · Dataset Setup"""

import yaml

data_yaml_candidates = list(INPUT_ROOT.rglob("data.yaml"))
if not data_yaml_candidates:
    raise RuntimeError(
        "Could not find data.yaml inside /kaggle/input\n"
        "Attach your pothole dataset via Add Data."
    )

selected_yaml = next(
    (p for p in data_yaml_candidates if "pothole" in str(p).lower()),
    data_yaml_candidates[0]
)

DATASET_DIR = selected_yaml.parent
print(f"Dataset detected: {DATASET_DIR}")

with open(selected_yaml, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(DATASET_DIR.resolve())
DATA_YAML = BASE_DIR / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

print(f"Writable data.yaml created: {DATA_YAML}")
for split in ["train", "val", "test"]:
    d = DATASET_DIR / "images" / split
    n = len(list(d.glob("*.*"))) if d.exists() else 0
    print(f"   {split}: {n} images")

print("=" * 60)
print("  [DATASET READY]")
print("=" * 60)


In [ ]:
"""
Section 2c · Checkpoint Import from YOLOv8l-Epochs dataset

Your attached YOLOv8l-Epochs dataset is at /kaggle/input/yolov8l-epochs/
It currently has: best.pt, last.pt, results.csv
That is enough to resume training.

If args.yaml is missing the notebook falls back to weight-only resume
automatically (no crash, no error — just epoch count resets).
"""

RUN_NAME    = "pothole_v8l"
weights_dir = RUNS_DIR / RUN_NAME / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
run_dir = RUNS_DIR / RUN_NAME

print("Searching for checkpoints in /kaggle/input ...")

last_pt_candidates = list(INPUT_ROOT.rglob("last.pt"))
best_pt_candidates = list(INPUT_ROOT.rglob("best.pt"))

if not last_pt_candidates and not best_pt_candidates:
    print("No checkpoint files found — training will start fresh.")
else:
    print(f"Found: {len(last_pt_candidates)} last.pt, {len(best_pt_candidates)} best.pt")
    all_ckpt = last_pt_candidates + best_pt_candidates
    ckpt_dir = all_ckpt[0].parent
    print(f"Checkpoint dir: {ckpt_dir}")

    for fname in ["last.pt", "best.pt"]:
        src = ckpt_dir / fname
        if src.is_file():
            shutil.copy2(src, weights_dir / fname)
            shutil.copy2(src, V8L_DIR / fname)
            mb = src.stat().st_size / 1e6
            print(f"  Copied {fname} ({mb:.1f} MB) -> weights dir + V8L_DIR")
        else:
            print(f"  {fname} not found in checkpoint dir")

    for fname in ["results.csv"]:
        src = ckpt_dir / fname
        if src.is_file():
            shutil.copy2(src, run_dir / fname)
            shutil.copy2(src, LOGS_DIR / f"{RUN_NAME}_{fname}")
            print(f"  Copied {fname} -> run dir")

    args_src = ckpt_dir / "args.yaml"
    if args_src.is_file():
        shutil.copy2(args_src, run_dir / "args.yaml")
        shutil.copy2(args_src, V8L_DIR / "args.yaml")
        print("  Copied args.yaml -> run dir + V8L_DIR (resume=True will work)")
    else:
        print("  args.yaml not in dataset — will use weight-only resume (epoch resets)")
        print("  NOTE: After this session, args.yaml will be auto-generated and pushed.")

print("\n" + "=" * 60)
print("  [CHECKPOINT IMPORT] Complete")
print("=" * 60)


In [ ]:
"""
Section 2d · Kaggle Dataset Auto-Push Configuration

HOW IT WORKS:
  Every 5 epochs, the training engine calls sync_checkpoints().
  sync_checkpoints() saves files locally AND pushes a new version
  of your YOLOv8l-Epochs Kaggle Dataset.

  Even if your session crashes, the dataset version from 5 epochs
  ago is permanently saved and you can resume from it next session.

ACTION REQUIRED:
  Set KAGGLE_DATASET_OWNER to your Kaggle username (lowercase).
  KAGGLE_DATASET_NAME must match the slug shown in the URL of your
  dataset, e.g. https://kaggle.com/datasets/USERNAME/yolov8l-epochs
                                                            ^
                                                   this part is the slug
"""

# ---- SET YOUR USERNAME HERE ----------------------------------------
KAGGLE_DATASET_OWNER = "your-kaggle-username"   # <- CHANGE THIS
KAGGLE_DATASET_NAME  = "yolov8l-epochs"          # <- match your dataset slug
# --------------------------------------------------------------------

KAGGLE_DATASET_REF = f"{KAGGLE_DATASET_OWNER}/{KAGGLE_DATASET_NAME}"
PUSH_STAGING_DIR   = BASE_DIR / "checkpoint_push_staging"
PUSH_STAGING_DIR.mkdir(exist_ok=True)

PUSH_ENABLED = (KAGGLE_DATASET_OWNER != "your-kaggle-username")

if PUSH_ENABLED:
    print(f"Kaggle auto-push ENABLED -> {KAGGLE_DATASET_REF}")
else:
    print("Kaggle auto-push DISABLED — set KAGGLE_DATASET_OWNER to enable.")
    print("Files will still be saved locally to /kaggle/working/ each sync.")


def push_to_kaggle_dataset(run_name: str, epoch: int = -1) -> bool:
    """
    Push checkpoint files to the Kaggle Dataset as a new version.
    Returns True on success, False on failure.
    Training always continues regardless of push success/failure.
    """
    if not PUSH_ENABLED:
        return False

    run_dir     = RUNS_DIR / run_name
    weights_dir = run_dir / "weights"

    # Clear staging dir
    for f in PUSH_STAGING_DIR.iterdir():
        if f.is_file(): f.unlink()

    files_pushed = []

    # Collect files -- prefer live weights dir, fall back to V8L_DIR
    for fname in ["last.pt", "best.pt"]:
        src = weights_dir / fname
        if not src.is_file(): src = V8L_DIR / fname
        if src.is_file():
            shutil.copy2(src, PUSH_STAGING_DIR / fname)
            files_pushed.append(fname)

    for fname in ["args.yaml"]:
        src = run_dir / fname
        if not src.is_file(): src = V8L_DIR / fname
        if src.is_file():
            shutil.copy2(src, PUSH_STAGING_DIR / fname)
            files_pushed.append(fname)

    for fname in ["results.csv"]:
        src = run_dir / fname
        if not src.is_file(): src = LOGS_DIR / f"{run_name}_{fname}"
        if src.is_file():
            shutil.copy2(src, PUSH_STAGING_DIR / fname)
            files_pushed.append(fname)

    if not files_pushed:
        print("  [KAGGLE PUSH] No files to push yet.")
        return False

    # push_info.json -- easy to see what epoch was saved
    push_info = {
        "run_name":      run_name,
        "epoch":         epoch,
        "files":         files_pushed,
        "pushed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    (PUSH_STAGING_DIR / "push_info.json").write_text(json.dumps(push_info, indent=2))

    # dataset-metadata.json (required by Kaggle API)
    meta = {
        "title": "YOLOv8l Epochs",
        "id":    KAGGLE_DATASET_REF,
        "licenses": [{"name": "other"}],
    }
    (PUSH_STAGING_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    notes = f"epoch={epoch} | {files_pushed}"
    cmd = [
        "kaggle", "datasets", "version",
        "--path",     str(PUSH_STAGING_DIR),
        "--message",  notes,
        "--dir-mode", "zip",
    ]

    print(f"  [KAGGLE PUSH] Uploading {files_pushed} -> {KAGGLE_DATASET_REF} ...")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            print(f"  [KAGGLE PUSH] New version created: epoch={epoch}, files={files_pushed}")
            return True
        else:
            print(f"  [KAGGLE PUSH] Failed (code={result.returncode})")
            if result.stderr.strip():
                print(f"               {result.stderr.strip()[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print("  [KAGGLE PUSH] Timed out (5 min). Training continues.")
        return False
    except Exception as e:
        print(f"  [KAGGLE PUSH] Error: {e}. Training continues.")
        return False


print(f"Push staging dir: {PUSH_STAGING_DIR}")


## 3 · GPU Detection & Dynamic Batch Sizing

In [ ]:
"""Dynamic GPU batch sizing via real memory profiling."""

GPU_TARGET_UTILIZATION = 0.85
MIN_BATCH   = 2
MAX_BATCH   = 64
IMGSZ_FALLBACKS = [768, 640, 512]


def _profile_model_memory(model_name: str, imgsz: int) -> tuple:
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats(0)
    model = YOLO(model_name)
    model.model.to("cuda").train()
    torch.cuda.reset_peak_memory_stats(0)
    d1 = torch.randn(1, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad(): _ = model.model(d1)
    m1 = torch.cuda.max_memory_allocated(0) / 1e9
    del d1, _; torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)
    d2 = torch.randn(2, 3, imgsz, imgsz, device="cuda", dtype=torch.float32)
    with torch.no_grad(): _ = model.model(d2)
    m2 = torch.cuda.max_memory_allocated(0) / 1e9
    del d2, _
    per_img = m2 - m1; base = m1 - per_img
    OVERHEAD = 2.2
    per_img *= OVERHEAD; base *= OVERHEAD
    del model; torch.cuda.empty_cache(); gc.collect()
    if per_img <= 0: per_img = m1 * 0.4; base = m1 * 0.6
    return max(base, 0.5), max(per_img, 0.05)


def estimate_batch_size(model_name: str, imgsz: int) -> tuple:
    if not torch.cuda.is_available():
        print("No GPU — defaulting to batch=2.")
        return MIN_BATCH, imgsz
    props = torch.cuda.get_device_properties(0)
    total = props.total_memory / 1e9
    target = total * GPU_TARGET_UTILIZATION
    print(f"GPU: {props.name} | VRAM: {total:.2f} GB | Target: {target:.2f} GB")
    for try_sz in ([imgsz] + [s for s in IMGSZ_FALLBACKS if s < imgsz]):
        try:
            print(f"\nProfiling {model_name} @ {try_sz}px...")
            base, per_img = _profile_model_memory(model_name, try_sz)
            print(f"   Base: {base:.2f} GB | Per-image: {per_img:.3f} GB")
            avail = target - base
            if avail <= 0: continue
            batch = max(MIN_BATCH, min(MAX_BATCH, int(avail / per_img)))
            pred  = base + batch * per_img
            print(f"   batch={batch} | Predicted: {pred:.2f} GB ({pred/total*100:.1f}%)")
            if try_sz != imgsz: print(f"   imgsz reduced {imgsz}->{try_sz}")
            return batch, try_sz
        except Exception as e:
            print(f"   Profiling failed at {try_sz}px: {e}")
            torch.cuda.empty_cache(); gc.collect()
    print("\nFallback: batch=2, imgsz=640")
    return MIN_BATCH, 640


bs_test, sz_test = estimate_batch_size("yolov8l.pt", 768)
print("\n" + "=" * 50)
print(f"  Final: batch={bs_test}, imgsz={sz_test}")
print("=" * 50)


## 4 · Training Engine

In [ ]:
"""
Core training engine -- YOLOv8l Kaggle edition.

Fixes vs old notebook:
  1. validate_checkpoint_health REMOVED -- was causing false "corrupted"
     on valid AMP checkpoints, renaming last.pt, restarting from epoch 17.
  2. amp=False -- prevents NaN loss at epoch ~54 with v8l at 768px.
  3. lr0=3e-4 -- conservative for large model (v8l > v8m).
  4. push_to_kaggle_dataset() called inside sync_checkpoints() every 5 epochs.
"""

PIPELINE_START_TIME      = time.time()
CHECKPOINT_SYNC_INTERVAL = 5


def elapsed_hours() -> float:
    return (time.time() - PIPELINE_START_TIME) / 3600


def find_resume_checkpoint(run_name: str, model_dir: Path) -> tuple:
    """
    Resume cascade -- no health-check, no file deletion.
    1. last.pt in run weights dir  -> resume=True
    2. last.pt in model_dir (V8L_DIR) -> copy -> resume=True
    3. best.pt in model_dir -> weight-only (epoch resets)
    4. Nothing -> fresh from yolov8l.pt
    """
    weights_dir = RUNS_DIR / run_name / "weights"

    last_pt = weights_dir / "last.pt"
    if last_pt.is_file():
        print("=" * 60)
        print("  [RESUME] last.pt found in weights dir -- resume=True")
        print(f"  Source: {last_pt}")
        print("=" * 60)
        return last_pt, True

    last_v8l = model_dir / "last.pt"
    if last_v8l.is_file():
        weights_dir.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(last_v8l, weights_dir / "last.pt")
            args_src = model_dir / "args.yaml"
            if args_src.is_file():
                shutil.copy2(args_src, RUNS_DIR / run_name / "args.yaml")
            print("=" * 60)
            print("  [RESUME] last.pt from V8L_DIR -> copied -> resume=True")
            print(f"  Source: {last_v8l}")
            print("=" * 60)
            return weights_dir / "last.pt", True
        except Exception as e:
            print(f"  Could not copy last.pt: {e}")

    best_pt = model_dir / "best.pt"
    if best_pt.is_file():
        print("=" * 60)
        print("  [RESUME] No last.pt -- using best.pt (weight-only, epoch resets)")
        print(f"  Source: {best_pt}")
        print("=" * 60)
        return best_pt, False

    print("=" * 60)
    print("  [RESUME] No checkpoint -- fresh start from yolov8l.pt")
    print("=" * 60)
    return None, False


def sync_checkpoints(run_name: str, model_dir: Path, current_epoch: int = -1) -> None:
    """Local sync + Kaggle Dataset push. Called every 5 epochs."""
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    if not weights.is_dir(): return

    for fname in ["last.pt", "best.pt"]:
        src = weights / fname
        if src.is_file():
            for dst in [model_dir, CHECKPOINTS_DIR]:
                try: shutil.copy2(src, dst / fname)
                except Exception as e: print(f"  Sync failed {fname}: {e}")

    for src in [run_dir / "results.csv"]:
        if src.is_file():
            try:
                shutil.copy2(src, LOGS_DIR / f"{run_name}_results.csv")
                shutil.copy2(src, model_dir / "results.csv")
            except Exception: pass

    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file():
        try: shutil.copy2(args_yaml, model_dir / "args.yaml")
        except Exception: pass

    print(f"  [CHECKPOINT SAVED] Synced locally (epoch={current_epoch})")
    push_to_kaggle_dataset(run_name, epoch=current_epoch)


def copy_stage_artifacts(run_name: str, model_dir: Path) -> None:
    run_dir = RUNS_DIR / run_name
    weights = run_dir / "weights"
    for fname in ["best.pt", "last.pt"]:
        src = weights / fname
        if src.is_file():
            shutil.copy2(src, model_dir / fname)
            shutil.copy2(src, CHECKPOINTS_DIR / fname)
            print(f"  {fname} -> {model_dir / fname}")
    for pattern in ["*.csv", "*.png", "*.json"]:
        for f in run_dir.glob(pattern):
            shutil.copy2(f, LOGS_DIR / f"{run_name}_{f.name}")
    args_yaml = run_dir / "args.yaml"
    if args_yaml.is_file(): shutil.copy2(args_yaml, model_dir / "args.yaml")
    print(f"  Logs -> {LOGS_DIR}")


def train_stage(
    model_name: str, imgsz: int, epochs: int, run_name: str,
    model_dir: Path, patience: int = 30, max_retries: int = 3,
) -> dict | None:
    batch, actual_imgsz = estimate_batch_size(model_name, imgsz)
    stage_start = time.time()

    for attempt in range(1, max_retries + 1):
        print("\n" + "=" * 65)
        print(f"  {run_name} | Attempt {attempt}/{max_retries} | "
              f"batch={batch} | imgsz={actual_imgsz}")
        print(f"  Elapsed: {elapsed_hours():.1f}h | Limit: {SESSION_TIME_LIMIT_H}h")
        print("=" * 65 + "\n")

        try:
            ckpt, use_resume = find_resume_checkpoint(run_name, model_dir)

            def _grad_clip_callback(trainer):
                if hasattr(trainer, "model"):
                    for p in trainer.model.parameters():
                        if p.grad is not None:
                            torch.nan_to_num_(p.grad, nan=0.0, posinf=1e4, neginf=-1e4)
                    torch.nn.utils.clip_grad_norm_(trainer.model.parameters(), max_norm=10.0)

            def _epoch_callback(trainer, _rn=run_name, _md=model_dir):
                loss = getattr(trainer, "loss", None)
                if loss is not None:
                    try:
                        if not torch.isfinite(torch.as_tensor(loss)).all():
                            print(f"\n[NaN GUARD] NaN/Inf loss at epoch {trainer.epoch} -- stopping!")
                            trainer.epoch = trainer.epochs
                            return
                    except Exception: pass

                if trainer.epoch % CHECKPOINT_SYNC_INTERVAL == 0 and trainer.epoch > 0:
                    sync_checkpoints(_rn, _md, current_epoch=trainer.epoch)

                h = elapsed_hours()
                remaining = SESSION_TIME_LIMIT_H - h
                if remaining < 0.5:
                    print("\n" + "!" * 60)
                    print(f"  [TIME GUARD] Only {remaining:.2f}h left -- saving and stopping!")
                    print("!" * 60)
                    sync_checkpoints(_rn, _md, current_epoch=trainer.epoch)
                    trainer.epoch = trainer.epochs
                elif remaining < 1.0:
                    print(f"\n  {remaining:.1f}h remaining before session limit")

            TRAIN_KWARGS = dict(
                data=str(DATA_YAML.resolve()),
                imgsz=actual_imgsz, epochs=epochs, patience=patience, batch=batch,
                optimizer="AdamW",
                lr0=3e-4,      # conservative for large model -- prevents NaN
                lrf=0.003,
                cos_lr=True, weight_decay=5e-4,
                warmup_epochs=10, warmup_bias_lr=0.01, momentum=0.937,
                amp=False,     # CRITICAL: disables AMP -- prevents NaN at epoch ~54
                # clip_grad=10.0,
                nbs=64,
                cache=False, workers=4,
                device=0 if torch.cuda.is_available() else "cpu",
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                degrees=5.0, translate=0.1, scale=0.5,
                shear=2.0, flipud=0.2, fliplr=0.5,
                mosaic=1.0, mixup=0.1, copy_paste=0.1,
                project=str(RUNS_DIR), name=run_name, exist_ok=True,
                save_period=5, plots=True, verbose=True,
            )

            if ckpt and use_resume:
                print("Attempting full resume (resume=True)...")
                try:
                    model = YOLO(str(ckpt))
                    model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                    model.add_callback("on_train_epoch_end", _epoch_callback)
                    results = model.train(resume=True)
                except Exception as resume_err:
                    print(f"  resume=True failed: {resume_err}")
                    print("  Falling back to weight-only load...")
                    torch.cuda.empty_cache(); gc.collect()
                    model = YOLO(str(ckpt))
                    model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                    model.add_callback("on_train_epoch_end", _epoch_callback)
                    results = model.train(**TRAIN_KWARGS)
            elif ckpt and not use_resume:
                print("Weight-only load from best.pt (epoch resets to 0)...")
                model = YOLO(str(ckpt))
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)
            else:
                print("Fresh training from yolov8l.pt...")
                model = YOLO(model_name)
                model.add_callback("on_before_optimizer_step", _grad_clip_callback)
                model.add_callback("on_train_epoch_end", _epoch_callback)
                results = model.train(**TRAIN_KWARGS)

            duration = (time.time() - stage_start) / 3600
            print(f"\n{run_name} completed in {duration:.2f} hours.")
            copy_stage_artifacts(run_name, model_dir)

            log_entry = {
                "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                "batch": batch, "duration_hours": round(duration, 2),
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            log_file = LOGS_DIR / "training_log.json"
            logs = json.loads(log_file.read_text()) if log_file.is_file() else []
            logs.append(log_entry)
            log_file.write_text(json.dumps(logs, indent=2))

            best_pt = model_dir / "best.pt"
            if best_pt.is_file():
                val_model = YOLO(str(best_pt))
                metrics = val_model.val(
                    data=str(DATA_YAML.resolve()), imgsz=actual_imgsz,
                    batch=8, device=0 if torch.cuda.is_available() else "cpu", verbose=False,
                )
                return {
                    "stage": run_name, "model": model_name, "imgsz": actual_imgsz,
                    "mAP50": metrics.box.map50, "mAP50_95": metrics.box.map,
                    "precision": metrics.box.mp, "recall": metrics.box.mr,
                    "duration_h": round(duration, 2),
                }
            return None

        except RuntimeError as e:
            err_str = str(e).lower()
            if "out of memory" in err_str or ("cuda" in err_str and "nan" not in err_str):
                print(f"\nCUDA OOM at batch={batch}, imgsz={actual_imgsz}")
                torch.cuda.empty_cache(); gc.collect()
                if batch > MIN_BATCH:
                    batch = max(MIN_BATCH, batch // 2)
                    print(f"  Reducing batch to {batch}")
                elif actual_imgsz > 640:
                    actual_imgsz -= 64; batch = MIN_BATCH
                    print(f"  Reducing imgsz to {actual_imgsz}")
                else:
                    print("  Cannot reduce further."); return None
            else:
                print(f"\nRuntimeError: {e}"); raise

    print(f"\n{run_name}: All {max_retries} attempts exhausted.")
    return None


print("Training engine loaded.")
print(f"  AMP     : DISABLED (amp=False)")
print(f"  LR      : lr0=3e-4")
print(f"  Resume  : no health-check, no file deletion")
print(f"  Sync    : every {5} epochs -> local + Kaggle Dataset push")
print(f"  Time    : {8.5}h limit")


## 5 · Train YOLOv8l (768px, 400 epochs)

In [ ]:
"""
Start or resume YOLOv8l training.

Safe to re-run. Automatically:
  1. Imports checkpoints from YOLOv8l-Epochs dataset (done in Section 2c)
  2. Resumes from last.pt (resume=True) if available
  3. Falls back to best.pt (weight-only) if last.pt missing
  4. Starts fresh if no checkpoint found
  5. Pushes checkpoints to YOLOv8l-Epochs every 5 epochs automatically
"""

print("\n" + "=" * 65)
print("  TRAINING -- YOLOv8l @ 768px")
print("=" * 65 + "\n")

metrics_v8l = train_stage(
    model_name = "yolov8l.pt",
    imgsz      = 768,
    epochs     = 400,
    run_name   = "pothole_v8l",
    model_dir  = V8L_DIR,
    patience   = 30,
)

if metrics_v8l:
    print("\nTraining Results:")
    for k, v in metrics_v8l.items():
        print(f"  {k:12s}: {v}")
else:
    print("\nTraining stopped early (time guard or session limit).")
    print("Checkpoint was auto-pushed to YOLOv8l-Epochs dataset.")
    print("Attach the updated dataset next session to resume.")

torch.cuda.empty_cache()
gc.collect()
print(f"\nTotal session time: {elapsed_hours():.1f}h")


## 6 · Evaluation

In [ ]:
"""Evaluate best model."""

metrics_local = globals().get("metrics_v8l", None)

if metrics_local is None and (V8L_DIR / "best.pt").is_file():
    print("Recovering metrics from best.pt...")
    try:
        val_model = YOLO(str(V8L_DIR / "best.pt"))
        m = val_model.val(
            data=str(DATA_YAML.resolve()), imgsz=768, batch=8,
            device=0 if torch.cuda.is_available() else "cpu", verbose=False,
        )
        metrics_local = {
            "mAP50": m.box.map50, "mAP50_95": m.box.map,
            "precision": m.box.mp, "recall": m.box.mr,
        }
    except Exception as e:
        print(f"  Recovery failed: {e}")

if metrics_local:
    print("YOLOv8l Results:")
    for k, v in metrics_local.items():
        if isinstance(v, float): print(f"  {k:>15s}: {v:.4f}")
        else: print(f"  {k:>15s}: {v}")

src = V8L_DIR / "best.pt"
if src.is_file():
    shutil.copy2(src, FINAL_DIR / "best.pt")
    print(f"\nbest.pt -> {FINAL_DIR / "best.pt"}")

BEST_MODEL_PATH = FINAL_DIR / "best.pt" if (FINAL_DIR / "best.pt").is_file() else V8L_DIR / "best.pt"
print(f"BEST_MODEL_PATH = {BEST_MODEL_PATH}")


## 7 · Final Push & Backup Download

Run this cell before stopping your session.

In [ ]:
"""
Final checkpoint push + backup download zip.

Run this before stopping your Kaggle session:
  1. Pushes final checkpoint to YOLOv8l-Epochs dataset
  2. Creates a local zip file as backup
  3. Shows a download link

The Kaggle Dataset push is the MAIN backup.
The zip is just extra safety in case something goes wrong.
"""

RUN_NAME = "pothole_v8l"

# Final push
print("Final push to Kaggle Dataset...")
sync_checkpoints(RUN_NAME, V8L_DIR, current_epoch=-1)

# Build zip
import time as _t
ts       = _t.strftime("%Y%m%d_%H%M%S")
zip_path = BASE_DIR / f"pothole_v8l_checkpoint_{ts}.zip"

run_dir     = RUNS_DIR / RUN_NAME
weights_dir = run_dir / "weights"

files_to_zip = {
    "last.pt":    [weights_dir / "last.pt",    V8L_DIR / "last.pt"],
    "best.pt":    [weights_dir / "best.pt",    V8L_DIR / "best.pt"],
    "args.yaml":  [run_dir / "args.yaml",      V8L_DIR / "args.yaml"],
    "results.csv":[run_dir / "results.csv",    LOGS_DIR / f"{RUN_NAME}_results.csv"],
}

print("\nCreating backup zip...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for fname, srcs in files_to_zip.items():
        for src in srcs:
            if Path(src).is_file():
                zf.write(src, fname)
                print(f"  {fname} ({Path(src).stat().st_size / 1e6:.1f} MB)")
                break
        else:
            print(f"  {fname} -- not found")

print(f"\nZip: {zip_path.name} ({zip_path.stat().st_size / 1e6:.1f} MB)")
print("\nBackup download link:")
display(FileLink(str(zip_path)))

print("\n" + "-" * 60)
print("To continue in Colab:")
print("  1. Go to your YOLOv8l-Epochs Kaggle Dataset -> Download")
print("  2. Extract and upload to Google Drive:")
print("     pothole_detection_system/backend/models/v8l/")
print("       last.pt, best.pt, args.yaml, results.csv")
print("  3. In Colab: run Section 2c then Section 5")
print("-" * 60)

# Epoch summary
for csv_path in [run_dir / "results.csv", LOGS_DIR / f"{RUN_NAME}_results.csv"]:
    if csv_path.is_file():
        try:
            df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
            ecol = "epoch" if "epoch" in df.columns else df.columns[0]
            last = df.iloc[-1]
            print(f"\nLast logged: epoch={int(last[ecol])}")
            for col, lbl in [("metrics/mAP50(B)","mAP@50"),("metrics/mAP50-95(B)","mAP@50-95")]:
                if col in df.columns: print(f"  {lbl}: {last[col]:.4f}")
        except Exception: pass
        break
